# Materialized Tables: Neo4j Aircraft Graph as Delta Tables

Materializes Neo4j graph data as **managed Delta tables** in Unity Catalog via UC JDBC,
making it queryable alongside `sensor_readings` with standard SQL.

**Why materialize?**
- **Genie / AI agents** can query graph data via natural language
- **Full SQL support** — no JDBC limitations at query time
- **Performance** — Delta tables are optimized for analytical queries
- **Governance** — materialized tables are visible in Catalog Explorer with full lineage

**Prerequisites:**
- Run `00-load-graph.ipynb` to load the aircraft graph into Neo4j
- Run `01-simple-connect-test.ipynb` to create the UC JDBC connection and `sensor_readings` table

## Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - Edit these values for your environment
# =============================================================================

# --- Neo4j Aura ---
NEO4J_URI = "neo4j+s://<instance>.databases.neo4j.io"
SECRET_SCOPE = "sample_validation"
NEO4J_USERNAME = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_USERNAME")
NEO4J_PASSWORD = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_PASSWORD")

# --- Databricks Unity Catalog ---
UC_CATALOG = "<catalog>"
UC_SCHEMA = "neo4j_getting_started"
UC_VOLUME = "aircraft_data"
JDBC_JAR_PATH = "/Volumes/<catalog>/<schema>/<volume>/neo4j-unity-catalog-connector.jar"
UC_CONNECTION_NAME = "sample_neo4j_jdbc_connection"

# =============================================================================
# DERIVED VALUES - no need to edit below this line
# =============================================================================
FQN = f"`{UC_CATALOG}`.`{UC_SCHEMA}`"
VOLUME_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{UC_VOLUME}"
NEO4J_HOST = NEO4J_URI.replace("neo4j+s://", "")
NEO4J_JDBC_URL_SQL = f"jdbc:neo4j+s://{NEO4J_HOST}:7687/neo4j?enableSQLTranslation=true"
JAVA_DEPENDENCIES = f'["{JDBC_JAR_PATH}"]'

print("Configuration:")
print(f"  Neo4j URI:       {NEO4J_URI}")
print(f"  Tables:          {FQN}.*")
print(f"  Volume:          {VOLUME_PATH}")
print(f"  JDBC JAR:        {JDBC_JAR_PATH}")
print(f"  UC Connection:   {UC_CONNECTION_NAME}")

In [ ]:
from pyspark.sql.types import StringType

def read_neo4j(custom_schema, query):
    """Read from Neo4j through the UC JDBC connection."""
    return (
        spark.read.format("jdbc")
        .option("databricks.connection", UC_CONNECTION_NAME)
        .option("customSchema", custom_schema)
        .option("query", query)
        .load()
    )

def materialize(table_name, query, custom_schema, expected):
    """Read from Neo4j via UC JDBC and write as a managed Delta table."""
    import time
    df = read_neo4j(custom_schema, query)
    # Cast all columns to STRING to avoid JDBC CHAR(0) Delta write errors
    for c in df.columns:
        df = df.withColumn(c, df[c].cast(StringType()))
    t0 = time.time()
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQN}.{table_name}")
    ms = (time.time() - t0) * 1000
    cnt = spark.sql(f"SELECT COUNT(*) AS cnt FROM {FQN}.{table_name}").collect()[0]["cnt"]
    status = "PASS" if cnt == expected else "FAIL"
    print(f"  [{status}] {table_name}: {cnt} rows ({ms:.0f}ms)")

---

## Section 1: Verify Data Sources

Confirms both `sensor_readings` (Delta) and the Neo4j aircraft graph (via JDBC) are accessible.

In [ ]:
print("--- Section 1: Verify Data Sources ---")

count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {FQN}.sensor_readings").collect()[0]["cnt"]
status = "PASS" if count == 172800 else "FAIL"
print(f"  [{status}] Delta sensor_readings: {count:,} rows")

for label, expected in [
    ("Aircraft", 20), ("Airport", 12), ("System", 80), ("Component", 320),
    ("Sensor", 160), ("Flight", 800), ("MaintenanceEvent", 300), ("Delay", 514),
]:
    cnt = read_neo4j("cnt LONG", f"SELECT COUNT(*) AS cnt FROM {label}").collect()[0]["cnt"]
    status = "PASS" if cnt == expected else "FAIL"
    print(f"  [{status}] Neo4j {label}: {cnt} nodes")

---

## Section 2: Materialize Neo4j Nodes as Delta Tables

Reads each node label from Neo4j via UC JDBC and writes it as a managed Delta table.
Also materializes the Aircraft → System relationship via a `NATURAL JOIN` aggregate.

In [ ]:
print("--- Section 2: Materialize Neo4j Data ---")

materialize("neo4j_aircraft",
    "SELECT aircraftId, tail_number, icao24, model, manufacturer, operator FROM Aircraft",
    "aircraftId STRING, tail_number STRING, icao24 STRING, model STRING, manufacturer STRING, operator STRING",
    20)

materialize("neo4j_airports",
    "SELECT airportId, name, city, country, iata, icao FROM Airport",
    "airportId STRING, name STRING, city STRING, country STRING, iata STRING, icao STRING",
    12)

materialize("neo4j_systems",
    "SELECT systemId, aircraftId, type, name FROM System",
    "systemId STRING, aircraftId STRING, type STRING, name STRING",
    80)

materialize("neo4j_sensors",
    "SELECT sensorId, systemId, type, name, unit FROM Sensor",
    "sensorId STRING, systemId STRING, type STRING, name STRING, unit STRING",
    160)

materialize("neo4j_components",
    "SELECT componentId, systemId, type, name FROM Component",
    "componentId STRING, systemId STRING, type STRING, name STRING",
    320)

materialize("neo4j_maintenance_events",
    "SELECT eventId, componentId, systemId, aircraftId, fault, severity, reported_at, corrective_action FROM MaintenanceEvent",
    "eventId STRING, componentId STRING, systemId STRING, aircraftId STRING, fault STRING, severity STRING, reported_at STRING, corrective_action STRING",
    300)

materialize("neo4j_flights",
    "SELECT flightId, flight_number, aircraftId, operator, origin, destination, scheduled_departure, scheduled_arrival FROM Flight",
    "flightId STRING, flight_number STRING, aircraftId STRING, operator STRING, origin STRING, destination STRING, scheduled_departure STRING, scheduled_arrival STRING",
    800)

materialize("neo4j_delays",
    "SELECT delayId, flightId, cause, CAST(minutes AS STRING) AS minutes FROM Delay",
    "delayId STRING, flightId STRING, cause STRING, minutes STRING",
    514)

# Relationship table via NATURAL JOIN
df = read_neo4j(
    "aircraftId STRING, model STRING, systemId STRING, systemType STRING, systemName STRING, cnt LONG",
    """SELECT a.aircraftId AS aircraftId, a.model AS model,
              s.systemId AS systemId, s.type AS systemType, s.name AS systemName,
              COUNT(*) AS cnt
       FROM Aircraft a NATURAL JOIN HAS_SYSTEM rel NATURAL JOIN System s
       GROUP BY a.aircraftId, a.model, s.systemId, s.type, s.name""",
).select("aircraftId", "model", "systemId", "systemType", "systemName")

for c in df.columns:
    df = df.withColumn(c, df[c].cast(StringType()))
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQN}.neo4j_aircraft_systems")
cnt = spark.sql(f"SELECT COUNT(*) AS cnt FROM {FQN}.neo4j_aircraft_systems").collect()[0]["cnt"]
status = "PASS" if cnt == 80 else "FAIL"
print(f"  [{status}] neo4j_aircraft_systems: {cnt} rows")

---

## Section 3: SQL Validation Tests

Verifies that standard SQL operations work on the materialized Delta tables.

In [ ]:
print("--- Section 3: SQL Validation Tests ---")

# TEST 1: GROUP BY — maintenance events by severity
print("\n  TEST 1: GROUP BY — maintenance events by severity")
spark.sql(f"""
    SELECT severity, COUNT(*) AS event_count
    FROM {FQN}.neo4j_maintenance_events
    GROUP BY severity
    ORDER BY event_count DESC
""").show(truncate=False)

# TEST 2: WHERE + ORDER BY — critical maintenance events
print("  TEST 2: WHERE + ORDER BY — critical events")
spark.sql(f"""
    SELECT eventId, aircraftId, fault, reported_at
    FROM {FQN}.neo4j_maintenance_events
    WHERE severity = 'CRITICAL'
    ORDER BY reported_at DESC
""").show(10, truncate=False)

# TEST 3: Aggregations + JOIN — sensor count per aircraft model
print("  TEST 3: Aggregations + JOIN — sensor count per model")
spark.sql(f"""
    SELECT a.model, a.manufacturer,
           COUNT(DISTINCT s.sensorId) AS sensor_count,
           COUNT(DISTINCT sys.systemId) AS system_count
    FROM {FQN}.neo4j_aircraft a
    JOIN {FQN}.neo4j_systems sys ON a.aircraftId = sys.aircraftId
    JOIN {FQN}.neo4j_sensors s ON sys.systemId = s.systemId
    GROUP BY a.model, a.manufacturer
    ORDER BY sensor_count DESC
""").show(truncate=False)

# TEST 4: DISTINCT — unique manufacturers and models
print("  TEST 4: DISTINCT — manufacturers and models")
spark.sql(f"""
    SELECT DISTINCT manufacturer, model
    FROM {FQN}.neo4j_aircraft
    ORDER BY manufacturer, model
""").show(truncate=False)

---

## Section 4: Federated Queries

Joins the materialized Neo4j tables with `sensor_readings` in pure SQL.
No JDBC or Python driver needed at query time — just Delta tables.

In [ ]:
print("--- Section 4: Federated Queries ---")

# Federated Query 1: Aircraft health overview
print("\n  Federated Query 1: Aircraft Health Overview")
spark.sql(f"""
    SELECT a.aircraftId, a.model, a.operator,
           COUNT(DISTINCT m.eventId) AS maintenance_events,
           SUM(CASE WHEN m.severity = 'CRITICAL' THEN 1 ELSE 0 END) AS critical_events,
           COUNT(DISTINCT s.sensorId) AS sensor_count,
           ROUND(AVG(r.value), 2) AS avg_sensor_reading
    FROM {FQN}.neo4j_aircraft a
    LEFT JOIN {FQN}.neo4j_maintenance_events m ON a.aircraftId = m.aircraftId
    LEFT JOIN {FQN}.neo4j_sensors s ON s.sensorId LIKE CONCAT(a.aircraftId, '-%')
    LEFT JOIN {FQN}.sensor_readings r ON r.sensor_id = s.sensorId
    GROUP BY a.aircraftId, a.model, a.operator
    ORDER BY critical_events DESC, maintenance_events DESC
""").show(10, truncate=False)

# Federated Query 2: Route analysis — busiest airport pairs
print("\n  Federated Query 2: Route Analysis")
spark.sql(f"""
    SELECT dep.city AS origin_city, dep.iata AS origin,
           arr.city AS destination_city, arr.iata AS destination,
           COUNT(*) AS flight_count
    FROM {FQN}.neo4j_flights f
    JOIN {FQN}.neo4j_airports dep ON f.origin = dep.iata
    JOIN {FQN}.neo4j_airports arr ON f.destination = arr.iata
    GROUP BY dep.city, dep.iata, arr.city, arr.iata
    ORDER BY flight_count DESC
    LIMIT 10
""").show(truncate=False)

# Federated Query 3: Sensor health by system type
print("\n  Federated Query 3: Sensor Health by System Type")
spark.sql(f"""
    SELECT sys.type AS system_type,
           COUNT(DISTINCT s.sensorId) AS sensor_count,
           COUNT(r.reading_id) AS total_readings,
           ROUND(AVG(r.value), 2) AS avg_reading,
           ROUND(STDDEV(r.value), 2) AS stddev_reading
    FROM {FQN}.neo4j_sensors s
    JOIN {FQN}.neo4j_systems sys ON s.systemId = sys.systemId
    JOIN {FQN}.sensor_readings r ON r.sensor_id = s.sensorId
    GROUP BY sys.type
    ORDER BY total_readings DESC
""").show(truncate=False)

# Federated Query 4: Delay analysis with airport and operator info
print("\n  Federated Query 4: Delay Analysis by Airport and Cause")
spark.sql(f"""
    SELECT dep.city AS departure_city, dep.iata,
           d.cause, COUNT(*) AS delay_count,
           ROUND(AVG(CAST(d.minutes AS INT)), 1) AS avg_delay_minutes
    FROM {FQN}.neo4j_delays d
    JOIN {FQN}.neo4j_flights f ON d.flightId = f.flightId
    JOIN {FQN}.neo4j_airports dep ON f.origin = dep.iata
    GROUP BY dep.city, dep.iata, d.cause
    ORDER BY delay_count DESC
    LIMIT 15
""").show(truncate=False)

print("\nStatus: PASS")

---

## Summary

All Neo4j graph data is now available as managed Delta tables in Unity Catalog.

| Table | Source | Rows |
|-------|--------|------|
| `neo4j_aircraft` | Aircraft nodes | 20 |
| `neo4j_airports` | Airport nodes | 12 |
| `neo4j_systems` | System nodes | 80 |
| `neo4j_sensors` | Sensor nodes | 160 |
| `neo4j_components` | Component nodes | 320 |
| `neo4j_maintenance_events` | MaintenanceEvent nodes | 300 |
| `neo4j_flights` | Flight nodes | 800 |
| `neo4j_delays` | Delay nodes | 514 |
| `neo4j_aircraft_systems` | Aircraft → HAS_SYSTEM → System | 80 |

**Next steps:**
- Add `sensor_readings` and all `neo4j_*` tables to a Genie space for natural language queries
- Schedule notebook Section 2 to refresh Neo4j data periodically